# Step 4 — Yearly publication counts, −5 through +5

Read the existing works cache and produce **one row per identified author**, anchored to the reference award year selected in step 2, retaining step-3 attributes and review flags.

## Counting rules
- Main yearly fields: `pubs_m5` … `pubs_m1`, `pubs_award_year`, `pubs_p1` … `pubs_p5`.
- Calendar year = reference award year + relative year. The award year (0) is stored separately and excluded from both five-year totals.
- Count distinct OpenAlex work IDs per author by their recorded `publication_year`. Co-authored works count once for each listed author, not fractionally. Do not deduplicate across authors.
- Include all cached work types, because this cache contains no work-type field. These are cached indexed-work counts, not a verified articles-only or proceedings-only measure. Different OpenAlex IDs for versions may remain separate.
- Do not inject an award paper if it is absent from the cache or force its publication year to equal its award year.
- Missing, empty, malformed, logged-failed, conflicting-key, or invalid-work cache entries are not turned into zero-publication histories. Keep observed valid-work counts separately; main counts remain missing for these authors pending review.
- For usable nonempty cache lists, zero means **no cached work in that year**, not independently verified zero actual publications. Cache pagination completeness and identity disambiguation remain unverified.
- Do not zero out pre-career years using potentially unreliable first-publication estimates.

## Cache limitation found in the older collector
It used cursor pagination and selected `id,publication_year,cited_by_count`, but a caught request exception could silently stop pagination and return a partial list. Even a recorded zero failures cannot prove completeness. This notebook makes no API calls and does not claim to repair missing collection.

Suggested location: `notebooks/04_yearly_publications.ipynb`. Run locally against the repository; this notebook has no fabricated real-data execution outputs.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib
import io
import json
import math
import pickle
import re
import shutil
import tempfile
import pandas as pd
from IPython.display import display

ROOT_OVERRIDE = None
FREEZE_REL = 'conf_paper/conf_data/frozen/award_cohort_v1'
STEP2_REL = 'conf_paper/conf_data/derived/reference_awards_v1'
STEP3_REL = 'conf_paper/conf_data/derived/author_attributes_v1'
CACHE_REL = 'conf_paper/conf_data/04_all_works_cache.pkl'
OUTPUT_VERSION = 'yearly_publications_v1'
WRITE_OUTPUTS = True
TRANSFORM_VERSION = 'yearly_publications_1.0'
OFFSETS = list(range(-5, 6))

YEAR_COLS = {k: ('pubs_m' + str(-k) if k < 0 else 'pubs_p' + str(k) if k > 0 else 'pubs_award_year') for k in OFFSETS}

def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha(p):
    return sha_bytes(p.read_bytes())

def csv(p):
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding='utf-8-sig')

def norm_id(v, prefix='A'):
    if not isinstance(v, str):
        return None
    m = re.fullmatch(r'(?:https?://openalex\.org/)?(' + prefix + r'\d+)/?', v.strip(), re.I)
    return m.group(1).upper() if m else None

def int_year(v):
    if isinstance(v, bool):
        return None
    try:
        y = float(v)
        return int(y) if math.isfinite(y) and y.is_integer() and 1 <= y <= datetime.now(timezone.utc).year else None
    except (ValueError, TypeError, OverflowError):
        return None

def bools(s):
    values = s.astype(str).str.strip().str.lower()
    if not set(values).issubset({'true', 'false', ''}):
        raise ValueError('Unexpected boolean values.')
    return values.eq('true')

def find_root():
    if ROOT_OVERRIDE:
        p = Path(ROOT_OVERRIDE).expanduser().resolve()
        if (p / STEP3_REL / 'manifest.json').is_file():
            return p
        raise FileNotFoundError('Configured root does not contain step-3 outputs.')
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (p / STEP3_REL / 'manifest.json').is_file():
            return p
    raise FileNotFoundError('Run inside the repository or set ROOT_OVERRIDE.')

def verify_outputs(folder):
    m = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    for name, expected in m['output_sha256'].items():
        p = folder / name
        if not p.is_file() or sha(p) != expected:
            raise RuntimeError('Input integrity check failed: ' + str(p) + '. Investigate changes or Git newline conversion; do not bypass.')
    return m

ROOT = find_root()
FROZEN, STEP2, STEP3 = (ROOT / p for p in (FREEZE_REL, STEP2_REL, STEP3_REL))
TARGET = ROOT / 'conf_paper/conf_data/derived' / OUTPUT_VERSION
freeze_meta = verify_outputs(FROZEN)
step2_meta = verify_outputs(STEP2)
step3_meta = verify_outputs(STEP3)
for name, expected in step2_meta['identity']['input_hashes'].items():
    if sha(FROZEN / name) != expected:
        raise RuntimeError('Step 2 references a different frozen input: ' + name)
for key, path in [('freeze_manifest', FROZEN / 'manifest.json'), ('step2_manifest', STEP2 / 'manifest.json')]:
    if step3_meta['identity']['input_hashes'][key] != sha(path):
        raise RuntimeError('Step 3 is not linked to current upstream outputs: ' + key)
attributes = csv(STEP3 / 'author_attributes.csv')
roster = csv(STEP2 / 'author_reference_roster.csv')
reference_links = csv(STEP2 / 'reference_year_award_links.csv')
assert attributes.author_id.is_unique and roster.author_id.is_unique
assert set(attributes.author_id) == set(roster.author_id)
assert attributes.author_id.map(norm_id).eq(attributes.author_id).all()
attributes['reference_award_year'] = attributes.reference_award_year.map(int_year)
roster['reference_award_year'] = roster.reference_award_year.map(int_year)
assert attributes.reference_award_year.notna().all()
assert attributes.reference_award_year.between(2000, 2018).all()
expected = attributes.author_id.map(roster.set_index('author_id').reference_award_year)
assert attributes.reference_award_year.eq(expected).all()
assert reference_links[['author_id', 'cohort_record_id']].duplicated().sum() == 0
assert set(reference_links.author_id) == set(roster.author_id)
reference_links['work_id_norm'] = reference_links.openalex_work_id.map(lambda v: norm_id(v, 'W'))
input_hashes = {'freeze_manifest': sha(FROZEN / 'manifest.json'),
                'step2_manifest': sha(STEP2 / 'manifest.json'),
                'step3_manifest': sha(STEP3 / 'manifest.json')}
print('Identified authors retained:', len(attributes))
print('Relative-year columns:', YEAR_COLS)

Identified authors retained: 2710
Relative-year columns: {-5: 'pubs_m5', -4: 'pubs_m4', -3: 'pubs_m3', -2: 'pubs_m2', -1: 'pubs_m1', 0: 'pubs_award_year', 1: 'pubs_p1', 2: 'pubs_p2', 3: 'pubs_p3', 4: 'pubs_p4', 5: 'pubs_p5'}


## 1. Restricted cache loading and author coverage

Only read a cache from your trusted repository. The loader forbids pickle global/class imports and persistent references. Do not replace it with unrestricted `pickle.load` if it fails. Missing authors will retain missing counts; we do not drop seniority-Unknown authors.

In [2]:
class BuiltinsOnlyUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global/class loading is forbidden.')
    def persistent_load(self, pid):
        raise pickle.UnpicklingError('Persistent references are forbidden.')

cache_path = ROOT / CACHE_REL
if not cache_path.is_file():
    raise FileNotFoundError('The works cache is missing. No publication histories will be invented.')
cache_bytes = cache_path.read_bytes()
input_hashes['works_cache'] = sha_bytes(cache_bytes)
expected_cache = step3_meta['identity']['input_hashes'].get('works_cache')
if expected_cache is None or input_hashes['works_cache'] != expected_cache:
    raise RuntimeError('Cache differs from step 3 or was not recorded there. Review/re-run step 3 against the intended cache before continuing.')
cache = BuiltinsOnlyUnpickler(io.BytesIO(cache_bytes)).load()
if not isinstance(cache, dict) or not isinstance(cache.get('all_works'), dict):
    raise ValueError('Unexpected cache schema; expected {all_works: dict, failed_authors: list}.')
raw_cache = cache['all_works']
key_groups = defaultdict(list)
unrecognized_key_count = 0
for key, values in raw_cache.items():
    aid = norm_id(key)
    if aid is None:
        unrecognized_key_count += 1
    else:
        key_groups[aid].append(values)
failed_ids = set()
failed_log = cache.get('failed_authors', [])
if not isinstance(failed_log, list):
    raise ValueError('Unexpected failed_authors schema.')
for entry in failed_log:
    raw_id = entry[0] if isinstance(entry, (tuple, list)) and entry else entry
    aid = norm_id(raw_id)
    if aid:
        failed_ids.add(aid)
expected_ids = set(attributes.author_id)
cache_overview = {
    'identified_authors_in_roster': len(expected_ids),
    'raw_cache_author_entries': len(raw_cache),
    'normalized_cache_author_ids': len(key_groups),
    'roster_authors_without_cache_key': len(expected_ids - set(key_groups)),
    'extra_cache_author_ids_not_in_roster': len(set(key_groups) - expected_ids),
    'unrecognized_cache_author_keys': unrecognized_key_count,
    'roster_authors_with_duplicate_normalized_keys': sum(len(key_groups[a]) > 1 for a in expected_ids & set(key_groups)),
    'roster_authors_in_failure_log': len(expected_ids & failed_ids)
}
display(pd.DataFrame(cache_overview.items(), columns=['measure', 'value']))

,measure,value
0,identified_authors_in_roster,2710
1,raw_cache_author_entries,2701
2,normalized_cache_author_ids,2701
3,roster_authors_without_cache_key,9
4,extra_cache_author_ids_not_in_roster,0
5,unrecognized_cache_author_keys,0
6,roster_authors_with_duplicate_normalized_keys,0
7,roster_authors_in_failure_log,0


## 2. Validate work IDs/years, deduplicate and expose conflicts

Duplicate occurrences of the same work ID with the same year count once within each author. Different years for the same work ID, including disagreements across authors, are flagged rather than arbitrarily resolved. Malformed/undated rows are preserved in an issues table, and invalidate that author's main count series until review.

`cached_valid_works.csv` retains available valid evidence even for flagged authors. It is not necessarily a complete publication history.

In [3]:
issue_rows = []
work_rows = []
audit_rows = []
reference_ids = {aid: set(g.work_id_norm.dropna()) for aid, g in reference_links.groupby('author_id')}
for author in attributes.itertuples(index=False):
    aid = author.author_id
    entries = key_groups.get(aid, [])
    reasons = []
    if not entries:
        reasons.append('missing_cache_key')
        works = None
    elif len(entries) > 1:
        reasons.append('duplicate_normalized_cache_keys')
        works = None
    else:
        works = entries[0]
    if aid in failed_ids:
        reasons.append('author_in_failure_log')
    if len(entries) == 1 and not isinstance(works, list):
        reasons.append('cache_entry_not_list')
        works = None
    if isinstance(works, list) and not works:
        reasons.append('empty_cache_list_not_verified_zero')
    raw_count = len(works) if isinstance(works, list) else None
    by_work = defaultdict(list)
    invalid_count = 0
    for index, w in enumerate(works or [], 1):
        if not isinstance(w, dict):
            invalid_count += 1
            issue_rows.append({'author_id': aid, 'cache_row': index, 'work_id': None,
                               'publication_year_raw': None, 'issue': 'work_entry_not_dict'})
            continue
        wid = norm_id(w.get('id'), 'W')
        year = int_year(w.get('publication_year'))
        if wid is None or year is None:
            invalid_count += 1
            issue_rows.append({'author_id': aid, 'cache_row': index, 'work_id': wid,
                               'publication_year_raw': str(w.get('publication_year')),
                               'issue': 'invalid_work_id_or_publication_year'})
            continue
        by_work[wid].append(year)
    if invalid_count:
        reasons.append('invalid_or_undated_work_rows')
    conflicting = 0
    duplicate_rows = 0
    for wid, years in by_work.items():
        ys = set(years)
        duplicate_rows += len(years) - 1
        if len(ys) != 1:
            conflicting += 1
            issue_rows.append({'author_id': aid, 'cache_row': None, 'work_id': wid,
                               'publication_year_raw': json.dumps(sorted(ys)), 'issue': 'conflicting_work_years_within_author'})
        for yr in sorted(ys):
            work_rows.append({'author_id': aid, 'work_id': wid, 'publication_year': yr,
                              'within_author_year_conflict': len(ys) != 1})
    if conflicting:
        reasons.append('conflicting_work_years_within_author')
    cached_ids = set(by_work)
    expected_refs = reference_ids.get(aid, set())
    missing_refs = sorted(expected_refs - cached_ids)
    audit_rows.append({'author_id': aid, 'raw_cached_work_rows': raw_count,
                       'distinct_valid_work_ids_before_global_audit': len(by_work),
                       'duplicate_work_rows_removed': duplicate_rows,
                       'invalid_work_rows': invalid_count,
                       'within_author_year_conflicts': conflicting,
                       'reference_work_ids_expected': len(expected_refs),
                       'reference_work_ids_missing_from_cache': len(missing_refs),
                       'missing_reference_work_ids_json': json.dumps(missing_refs),
                       'cache_issues': ' | '.join(sorted(set(reasons))),
                       'pagination_completeness_verified': False})
work_columns = ['author_id', 'work_id', 'publication_year', 'within_author_year_conflict']
works_df = pd.DataFrame(work_rows, columns=work_columns)
audit = pd.DataFrame(audit_rows)
if not works_df.empty:
    global_year_counts = works_df.groupby('work_id').publication_year.nunique()
    conflict_ids = set(global_year_counts[global_year_counts.gt(1)].index)
    works_df['global_work_year_conflict'] = works_df.work_id.isin(conflict_ids)
    conflicted_author_ids = set(works_df.loc[works_df.global_work_year_conflict, 'author_id'])
    for wid, g in works_df.loc[works_df.global_work_year_conflict].groupby('work_id'):
        for aid in g.author_id.unique():
            issue_rows.append({'author_id': aid, 'cache_row': None, 'work_id': wid,
                               'publication_year_raw': json.dumps(sorted(map(int, g.publication_year.unique()))),
                               'issue': 'conflicting_work_years_across_cache'})
else:
    conflict_ids = set()
    conflicted_author_ids = set()
    works_df['global_work_year_conflict'] = pd.Series(dtype=bool)
for idx in audit.index[audit.author_id.isin(conflicted_author_ids)]:
    audit.loc[idx, 'cache_issues'] = ' | '.join(filter(None, [audit.loc[idx, 'cache_issues'], 'conflicting_work_years_across_cache']))
valid_works = works_df.loc[~works_df.within_author_year_conflict.astype(bool) & ~works_df.global_work_year_conflict.astype(bool)].copy()
assert not valid_works[['author_id', 'work_id']].duplicated().any()
valid_counts = valid_works.groupby('author_id').size()
audit['distinct_valid_dated_works_counted'] = audit.author_id.map(valid_counts).fillna(0).astype(int)
audit['counts_usable_as_provisional_cache_counts'] = audit.cache_issues.eq('')
audit['publication_counts_provisional'] = True

# Missing reference works are review flags, not grounds to insert works or dates.
audit['cache_requires_followup'] = ~audit.counts_usable_as_provisional_cache_counts | audit.reference_work_ids_missing_from_cache.gt(0)
work_issues = pd.DataFrame(issue_rows, columns=['author_id', 'cache_row', 'work_id', 'publication_year_raw', 'issue'])
display(audit.loc[audit.cache_requires_followup].head(30))
print('Distinct work IDs with conflicting years anywhere in the roster cache:', len(conflict_ids))

,author_id,raw_cached_work_rows,distinct_valid_work_ids_before_global_audit,duplicate_work_rows_removed,invalid_work_rows,within_author_year_conflicts,reference_work_ids_expected,reference_work_ids_missing_from_cache,missing_reference_work_ids_json,cache_issues,pagination_completeness_verified,distinct_valid_dated_works_counted,counts_usable_as_provisional_cache_counts,publication_counts_provisional,cache_requires_followup
18,A5000796744,30.0,29,0,1,0,1,0,[],invalid_or_undated_work_rows,False,29,False,True,True
32,A5001534492,269.0,268,0,1,0,1,0,[],invalid_or_undated_work_rows,False,268,False,True,True
74,A5003040843,653.0,652,0,1,0,1,0,[],invalid_or_undated_work_rows,False,652,False,True,True
104,A5004444958,271.0,270,0,1,0,1,0,[],invalid_or_undated_work_rows,False,270,False,True,True
113,A5004988074,30.0,29,0,1,0,1,0,[],invalid_or_undated_work_rows,False,29,False,True,True
132,A5005715220,7.0,7,0,0,0,1,1,"[""W2109992755""]",,False,7,True,True,True
133,A5005730180,310.0,309,0,1,0,1,0,[],invalid_or_undated_work_rows,False,309,False,True,True
145,A5006126917,29.0,28,0,1,0,1,0,[],invalid_or_undated_work_rows,False,28,False,True,True
153,A5006533777,284.0,283,0,1,0,1,0,[],invalid_or_undated_work_rows,False,283,False,True,True
204,A5008961593,462.0,461,0,1,0,1,0,[],invalid_or_undated_work_rows,False,461,False,True,True


Distinct work IDs with conflicting years anywhere in the roster cache: 0


## 3. Count all eleven relative years

Keep a tidy author/year table and a wide author table. `publications_cached_observed` includes valid dated evidence even when an author is flagged; `publications` is blank for unusable cache histories. Neither is represented as independently verified complete output.

The award year remains in the long table as relative year 0 and in the wide table as `pubs_award_year`. Five-year pre/post totals require all five main counts to be present.

In [4]:
counts = valid_works.groupby(['author_id', 'publication_year']).size().to_dict()
audit_lookup = audit.set_index('author_id').to_dict('index')
long_rows = []
wide_rows = []
for a in attributes.itertuples(index=False):
    aid = a.author_id
    ref_year = int(a.reference_award_year)
    state = audit_lookup[aid]
    usable = state['counts_usable_as_provisional_cache_counts']
    # A missing/empty/bad cache has no asserted zeros. Valid partial evidence remains visible.
    has_valid_evidence = state['distinct_valid_dated_works_counted'] > 0
    wide = {'author_id': aid}
    for k in OFFSETS:
        calendar_year = ref_year + k
        observed = int(counts.get((aid, calendar_year), 0)) if has_valid_evidence else None
        value = observed if usable else None
        name = YEAR_COLS[k]
        wide[name] = value
        wide[name + '_cached_observed'] = observed
        long_rows.append({'author_id': aid, 'reference_award_year': ref_year,
                          'relative_year': k, 'calendar_year': calendar_year,
                          'period': 'before' if k < 0 else 'after' if k > 0 else 'award_year',
                          'publications': value, 'publications_cached_observed': observed,
                          'count_status': 'cached_observed_completeness_unverified' if usable else 'missing_or_flagged_cache_history',
                          'cache_issues': state['cache_issues'],
                          'pagination_completeness_verified': False,
                          'publication_counts_provisional': True})
    wide_rows.append(wide)
long = pd.DataFrame(long_rows)
wide = pd.DataFrame(wide_rows)
for c in ['publications', 'publications_cached_observed']:
    long[c] = long[c].astype('Int64')
count_cols = [YEAR_COLS[k] for k in OFFSETS]
for c in count_cols + [c + '_cached_observed' for c in count_cols]:
    wide[c] = wide[c].astype('Int64')
before_cols = [YEAR_COLS[k] for k in range(-5, 0)]
after_cols = [YEAR_COLS[k] for k in range(1, 6)]
wide['pubs_before_5y'] = wide[before_cols].sum(axis=1, min_count=5).astype('Int64')
wide['pubs_after_5y'] = wide[after_cols].sum(axis=1, min_count=5).astype('Int64')
wide['pubs_total_11y'] = wide[count_cols].sum(axis=1, min_count=11).astype('Int64')
wide['pubs_before_mean_per_year'] = wide.pubs_before_5y / 5
wide['pubs_after_mean_per_year'] = wide.pubs_after_5y / 5

# Preserve prior attributes and Unknown seniority labels; do not reselect reference awards.
collisions = (set(wide) | set(audit)) & set(attributes) - {'author_id'}
if collisions:
    raise ValueError('Unexpected overlapping output columns: ' + repr(sorted(collisions)))
master = attributes.merge(audit, on='author_id', how='left', validate='one_to_one').merge(wide, on='author_id', how='left', validate='one_to_one')
assert master.author_id.is_unique and len(master) == len(attributes)
assert len(long) == len(attributes) * 11
assert not long[['author_id', 'relative_year']].duplicated().any()
assert long.groupby('author_id').relative_year.nunique().eq(11).all()
assert long.calendar_year.eq(long.reference_award_year + long.relative_year).all()
assert long.publications.dropna().ge(0).all()
assert long.loc[long.count_status.eq('missing_or_flagged_cache_history'), 'publications'].isna().all()
checkable = master[count_cols].notna().all(axis=1)
assert (master.loc[checkable, 'pubs_before_5y'] + master.loc[checkable, 'pubs_award_year'] + master.loc[checkable, 'pubs_after_5y']).eq(master.loc[checkable, 'pubs_total_11y']).all()
for k, c in YEAR_COLS.items():
    indexed = long.loc[long.relative_year.eq(k)].set_index('author_id').publications
    actual = master.set_index('author_id')[c]
    pd.testing.assert_series_equal(actual.sort_index(), indexed.sort_index(), check_names=False)
display(master[['author_id', 'author_name', 'reference_award_year', *count_cols,
                'pubs_before_5y', 'pubs_after_5y', 'cache_issues']].head(20))

,author_id,author_name,reference_award_year,pubs_m5,pubs_m4,pubs_m3,pubs_m2,pubs_m1,pubs_award_year,pubs_p1,pubs_p2,pubs_p3,pubs_p4,pubs_p5,pubs_before_5y,pubs_after_5y,cache_issues
0,A5000009841,Ana Klimovic,2014,0,0,0,0,1,1,0,3,6,3,2,1,14,
1,A5000102893,David Akers,2009,1,2,2,1,0,1,0,0,1,0,0,6,1,
2,A5000107991,Michael Vrable,2008,0,0,4,1,0,0,2,3,2,3,0,5,10,
3,A5000163896,Elisa D. Mekler,2016,0,2,3,2,5,13,4,15,9,9,13,12,50,
4,A5000167011,I. Scott MacKenzie,2008,8,4,3,8,10,8,13,9,4,9,12,33,47,
5,A5000170074,Alexandru Agache,2017,2,0,0,2,0,2,1,0,1,0,0,4,2,
6,A5000245150,Alexander J. Smola,2014,2,5,8,8,6,10,13,13,22,4,6,29,58,
7,A5000246567,Matthew L. Lee,2014,0,0,0,0,0,2,0,0,0,0,0,0,0,
8,A5000294737,Philipp Schwartenbeck,2016,0,0,3,5,3,6,0,5,4,6,4,11,19,
9,A5000308912,Jaechang Nam,2013,0,2,0,2,0,4,1,4,1,2,3,4,11,


## 4. Audit reference-paper coverage and summarise

A missing reference paper can reveal stale/misassigned IDs or collection gaps. A different publication year may be legitimate award/publication timing; flag it rather than moving the work to year 0. Preserve every same-year award-paper link in this check.

In [5]:
work_lookup = valid_works.set_index(['author_id', 'work_id']).publication_year.to_dict()
reference_audit = reference_links[[c for c in ['author_id', 'cohort_record_id', 'award_year', 'reference_award_year', 'openalex_work_id', 'work_id_norm'] if c in reference_links]].copy()
reference_audit['work_publication_year_in_cache'] = pd.Series([
    work_lookup.get((a, w)) for a, w in zip(reference_audit.author_id, reference_audit.work_id_norm)
], index=reference_audit.index, dtype='Int64')
reference_audit['reference_year_from_roster'] = reference_audit.author_id.map(roster.set_index('author_id').reference_award_year)
reference_audit['reference_work_missing_or_conflicting_in_cache'] = reference_audit.work_publication_year_in_cache.isna()
reference_audit['publication_year_differs_from_reference_year'] = (reference_audit.work_publication_year_in_cache.ne(reference_audit.reference_year_from_roster)).astype('boolean')
coverage = long.groupby(['relative_year', 'calendar_year'], dropna=False).agg(
    author_rows=('author_id', 'size'),
    authors_with_main_count=('publications', 'count'),
    authors_with_observed_evidence=('publications_cached_observed', 'count')).reset_index()
summary_values = {
    **cache_overview,
    'authors_with_usable_provisional_count_series': int(audit.counts_usable_as_provisional_cache_counts.sum()),
    'authors_with_missing_main_count_series': int((~audit.counts_usable_as_provisional_cache_counts).sum()),
    'authors_requiring_cache_followup': int(audit.cache_requires_followup.sum()),
    'duplicate_work_rows_removed_within_author': int(audit.duplicate_work_rows_removed.sum()),
    'invalid_or_undated_cached_work_rows': int(audit.invalid_work_rows.sum()),
    'distinct_author_work_pairs_with_valid_unambiguous_year': len(valid_works),
    'work_ids_with_conflicting_years': len(conflict_ids),
    'author_year_rows_expected': len(attributes) * 11,
    'author_year_rows_exported': len(long),
    'reference_links_missing_or_conflicting_in_cache': int(reference_audit.reference_work_missing_or_conflicting_in_cache.sum()),
    'reference_links_with_different_publication_year': int(reference_audit.publication_year_differs_from_reference_year.fillna(False).sum()),
    'authors_with_independently_verified_pagination_completeness': 0
}
summary = pd.DataFrame(summary_values.items(), columns=['measure', 'value'])
cache_review = audit.loc[audit.cache_requires_followup].merge(attributes[['author_id', 'author_name', 'reference_award_year']], on='author_id', validate='one_to_one')
display(summary)
display(reference_audit.loc[reference_audit.reference_work_missing_or_conflicting_in_cache | reference_audit.publication_year_differs_from_reference_year.fillna(False)].head(30))

,measure,value
0,identified_authors_in_roster,2710
1,raw_cache_author_entries,2701
2,normalized_cache_author_ids,2701
3,roster_authors_without_cache_key,9
4,extra_cache_author_ids_not_in_roster,0
5,unrecognized_cache_author_keys,0
6,roster_authors_with_duplicate_normalized_keys,0
7,roster_authors_in_failure_log,0
8,authors_with_usable_provisional_count_series,2593
9,authors_with_missing_main_count_series,117


,author_id,cohort_record_id,award_year,reference_award_year,openalex_work_id,work_id_norm,work_publication_year_in_cache,reference_year_from_roster,reference_work_missing_or_conflicting_in_cache,publication_year_differs_from_reference_year
2,A5000107991,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2008,2008,W2013592197,W2013592197,2010,2008,False,True
10,A5000416532,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2011,2011,W4230948688,W4230948688,2012,2011,False,True
15,A5000660268,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2012,2012,W2953102514,W2953102514,2013,2012,False,True
20,A5000876822,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2015,2015,W2580339477,W2580339477,2017,2015,False,True
23,A5000977391,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2005,2005,W2047418149,W2047418149,2006,2005,False,True
36,A5001597905,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2014,2014,W2061643296,W2061643296,2015,2014,False,True
39,A5001618396,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2012,2012,W1995509853,W1995509853,2014,2012,False,True
42,A5001760915,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2003,2003,W2056860348,W2056860348,2005,2003,False,True
49,A5002059431,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2011,2011,W4230948688,W4230948688,2012,2011,False,True
61,A5002433369,566ad7fa68317dc9d1dc2f15d68b93c0d0513f49439421...,2006,2006,W1981420413,W1981420413,2008,2006,False,True


## 5. Export versioned outputs without modifying upstream data

The main one-row-per-author output is `author_yearly_publications.csv`. Do not interpret blank main counts as zeros. Keep cache-audit and provisional flags when joining into the eventual master CSV.

The collection history does not establish pagination completeness, so final analyses should report this limitation and prioritise cache recollection/validation where needed. No citation totals or lift significance tests are calculated in this step.

In [6]:
settings = {'transform_version': TRANSFORM_VERSION, 'offsets': OFFSETS,
            'count_unit': 'distinct_OpenAlex_work_ID_per_author', 'type_filter': 'none; type unavailable in cache',
            'award_year_separate': True, 'count_conflicts': 'main author series missing; retain valid observed evidence',
            'pagination_completeness': 'unverified for legacy cache', 'cache_collection_date': 'not reliably recorded'}
identity = {'input_hashes': input_hashes, 'settings': settings}
if WRITE_OUTPUTS:
    if TARGET.exists():
        previous = json.loads((TARGET / 'manifest.json').read_text(encoding='utf-8'))
        if previous['identity'] != identity:
            raise RuntimeError('Existing step-4 inputs/settings differ. Review and choose a new OUTPUT_VERSION.')
        for name, h in previous['output_sha256'].items():
            if not (TARGET / name).is_file() or sha(TARGET / name) != h:
                raise RuntimeError('Existing step-4 output integrity failure: ' + name)
        print('Verified existing step-4 outputs; nothing overwritten:', TARGET)
    else:
        TARGET.parent.mkdir(parents=True, exist_ok=True)
        stage = Path(tempfile.mkdtemp(prefix='.' + OUTPUT_VERSION + '_', dir=TARGET.parent))
        try:
            tables = {'author_yearly_publications.csv': master,
                      'yearly_publications_long.csv': long,
                      'author_cache_audit.csv': audit,
                      'cache_review.csv': cache_review,
                      'cached_valid_works.csv': valid_works,
                      'work_issues.csv': work_issues,
                      'reference_work_cache_audit.csv': reference_audit,
                      'yearly_count_coverage.csv': coverage,
                      'step4_summary.csv': summary}
            for name, frame in tables.items():
                frame.to_csv(stage / name, index=False)
            readback = csv(stage / 'author_yearly_publications.csv')
            longback = csv(stage / 'yearly_publications_long.csv')
            assert len(readback) == len(attributes) and readback.author_id.is_unique
            assert len(longback) == 11 * len(attributes)
            dictionary = {'yearly_columns': {str(k): c for k, c in YEAR_COLS.items()},
                          'pubs_mN': 'Distinct cached works in reference year minus N; blank if cache series unusable.',
                          'pubs_award_year': 'Distinct cached works in the reference award calendar year (relative year zero).',
                          'pubs_pN': 'Distinct cached works in reference year plus N; blank if cache series unusable.',
                          '*_cached_observed': 'Valid unambiguous cached evidence even for some flagged authors; not a completeness claim.',
                          'pubs_before_5y': 'Sum m5 through m1 only; award year excluded; blank unless all five counts present.',
                          'pubs_after_5y': 'Sum p1 through p5 only; award year excluded; blank unless all five counts present.',
                          'zero': 'No work observed in a usable nonempty cached author list for that year; not independently verified actual zero.',
                          'missing': 'Unavailable/empty/flagged cache history, not zero publications.',
                          'count_scope': 'All cached work types. Distinct IDs, not bibliographically merged versions. Full per-author counting.',
                          'provisional': 'Legacy pagination, author identity and reference-event completeness are not independently validated.'}
            (stage / 'data_dictionary.json').write_text(json.dumps(dictionary, indent=2), encoding='utf-8')
            (stage / 'README.md').write_text(
                '# Step 4: yearly publications\n\n'
                'One author row plus 11 relative-year rows per identified author, anchored to step 2 reference awards. '
                'Award year is separate from five-year pre/post totals. Count distinct cached OpenAlex work IDs. '
                'Do not treat blanks as zeros or cached observed counts as independently complete histories. '
                'The legacy collector could silently return partial lists. No APIs, profile metrics, seniority reselection, '
                'citation calculations or upstream edits were performed. Review step4_summary.csv, cache_review.csv, '
                'work_issues.csv and reference_work_cache_audit.csv before interpreting results.\n', encoding='utf-8')
            hashes = {p.name: sha(p) for p in sorted(stage.iterdir()) if p.is_file()}
            manifest = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'identity': identity,
                        'output_sha256': hashes, 'external_api_calls': False, 'upstream_files_modified': False,
                        'summary': {k: int(v) for k, v in summary_values.items()}}
            (stage / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
            if TARGET.exists():
                raise FileExistsError('Output directory appeared during export; refusing overwrite.')
            stage.rename(TARGET)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
        print('Created and checked step-4 outputs:', TARGET)
else:
    print('Preview only. Set WRITE_OUTPUTS=True to export.')

Created and checked step-4 outputs: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\derived\yearly_publications_v1
